In [45]:
import os
import csv
import sys
from datetime import datetime, timedelta
import requests
from bs4 import BeautifulSoup

# 콘솔 출력 인코딩 설정 (Windows 환경의 한글 깨짐 방지)
if sys.platform.startswith('win'):
    try:
        sys.stdout.reconfigure(encoding='utf-8')
    except AttributeError:
        pass

In [46]:
# 주요 부동산 정책 데이터 정의 (이재명 대통령 6.27 대책만 타겟팅)
POLICIES = [
    {
        "president": "이재명",
        "policy": "6·27 가계부채 관리 강화방안",
        "announcement_date": "2025-06-27",
        "effective_date": "2025-06-28",
        "summary": "수도권 주담대 규제 강화"
    }
]

In [47]:
def get_policy_mapping(date_str):
    """
    입력된 날짜에 대응하는 부동산 정책 정보 및 시기(시행전, 시행일, 초기반응, 체감반응)를 조회합니다.

    Args:
        date_str (str): YYYYMMDD 또는 YYYY-MM-DD 형식의 날짜 문자열

    Returns:
        list: 매칭된 정책 정보 딕셔너리 리스트. 매칭되는 정책이 없을 경우 빈 리스트를 반환합니다.
    """
    # 날짜 파싱 시도
    input_date = None
    for fmt in ("%Y%m%d", "%Y-%m-%d"):
        try:
            input_date = datetime.strptime(date_str, fmt).date()
            break
        except ValueError:
            continue

    if not input_date:
        print(f"[Warning] 날짜 포맷이 올바르지 않습니다: {date_str}")
        return []

    matched = []

    for item in POLICIES:
        eff_date = datetime.strptime(item["effective_date"], "%Y-%m-%d").date()

        # 시기 정의 계산
        before_start = eff_date - timedelta(days=30)
        before_end = eff_date - timedelta(days=1)
        after_start = eff_date + timedelta(days=1)
        after_end = eff_date + timedelta(days=30)
        feel_start = eff_date + timedelta(days=31)
        feel_end = eff_date + timedelta(days=90)

        period = None
        if before_start <= input_date <= before_end:
            period = "시행전"
        elif input_date == eff_date:
            period = "시행일"
        elif after_start <= input_date <= after_end:
            period = "초기반응"
        elif feel_start <= input_date <= feel_end:
            period = "체감반응"

        if period:
            matched.append({
                "president": item["president"],
                "policy": item["policy"],
                "policy_summary": item["summary"],
                "period": period,
                "date": input_date.strftime("%Y-%m-%d")
            })

    return matched

In [48]:
import time
def scrape_naver_land_news(date_str, max_articles=100):
    """
    네이버 부동산 뉴스 섹션에서 특정 날짜의 기사를 페이지네이션(더보기 API 호출)을 통해
    최대 max_articles개 수집합니다. (하루 최대 100개 스크랩)
    """
    headers = {
        'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36'
    }
    articles = []
    
    url = f"https://news.naver.com/breakingnews/section/101/260?date={date_str}"
    try:
        res = requests.get(url, headers=headers)
        if not res.ok:
            # 에러가 발생해도 빈 리스트를 반환하여 루프가 중단되지 않도록 함
            return articles
            
        soup = BeautifulSoup(res.text, 'html.parser')
        a_tags = soup.select("a.sa_text_title")
        for a in a_tags:
            title = a.text.strip()
            link = a.get('href', '').strip()
            if title and link:
                articles.append((title, link))
                
        if len(articles) >= max_articles:
            return articles[:max_articles]
            
        container = soup.select_one("div.section_latest_article._CONTENT_LIST._PERSIST_META")
        if not container:
            return articles
            
        cursor = container.get("data-cursor")
        has_next = container.get("data-has-next")
        
        page_no = 2
        while cursor and has_next == "true" and len(articles) < max_articles:
            api_url = f"https://news.naver.com/section/template/SECTION_ARTICLE_LIST_FOR_LATEST?sid=101&sid2=260&cluid=&pageNo={page_no}&date={date_str}&next={cursor}"
            api_res = requests.get(api_url, headers=headers)
            if not api_res.ok:
                break
                
            data = api_res.json()
            rendered_component = data.get("renderedComponent", {})
            html_content = rendered_component.get("SECTION_ARTICLE_LIST_FOR_LATEST", "")
            if not html_content:
                break
                
            api_soup = BeautifulSoup(html_content, 'html.parser')
            api_a_tags = api_soup.select("a.sa_text_title")
            if not api_a_tags:
                break
                
            for a in api_a_tags:
                title = a.text.strip()
                link = a.get('href', '').strip()
                if title and link:
                    articles.append((title, link))
                    
            if len(articles) >= max_articles:
                return articles[:max_articles]
                
            cursor_el = api_soup.select_one("div.section_latest_article._CONTENT_LIST._PERSIST_META")
            if not cursor_el:
                break
                
            cursor = cursor_el.get("data-cursor")
            has_next = cursor_el.get("data-has-next")
            page_no += 1
            
            time.sleep(0.1) # 속도를 위해 딜레이 단축 (0.1초)
            
    except Exception as e:
        pass # 에러는 무시하고 수집된 만큼 반환
        
    return articles

In [49]:
def save_to_csv(rows, filename="data/News_Scraping_retouch.csv"):
    """
    가공된 뉴스 데이터를 CSV 파일에 누적 저장하며, 중복된 데이터는 저장하지 않습니다.
    
    중복 판단 기준은 (정책명, 날짜, 기사 URL)의 조합입니다.

    Args:
        rows (list): [대통령, 정책, 정책요약, 시기, 날짜, 기사제목, url, 감정] 형태의 데이터 리스트
        filename (str): 저장할 CSV 파일 이름
    """
    file_exists = os.path.exists(filename)
    existing_keys = set()
    
    # 1. 파일이 이미 존재하면 기존에 등록된 키 값을 읽어서 중복 판단용 set을 구축
    if file_exists:
        try:
            with open(filename, mode='r', encoding='utf-8-sig') as f:
                reader = csv.reader(f)
                header = next(reader, None)
                if header:
                    for line in reader:
                        if len(line) >= 7:
                            policy = line[1]
                            date = line[4]
                            url = line[6]
                            existing_keys.add((policy, date, url))
        except Exception as e:
            print(f"[Warning] 기존 CSV 파일을 읽는 동안 오류가 발생했습니다: {e}")
            
    # 2. 이번에 추가할 행들 중 중복되지 않는 고유한 행만 선별
    new_rows = []
    for r in rows:
        policy = r[1]
        date = r[4]
        url = r[6]
        key = (policy, date, url)
        if key not in existing_keys:
            new_rows.append(r)
            existing_keys.add(key)
            
    if not new_rows:
        print(f"[Info] 추가할 새로운 뉴스 데이터가 없습니다 (모든 데이터가 '{filename}'에 이미 존재함).")
        return

    # 3. 새로운 행들만 추가 저장
    try:
        file_exists = os.path.exists(filename)
        with open(filename, mode='a', encoding='utf-8-sig', newline='') as f:
            writer = csv.writer(f)
            if not file_exists:
                writer.writerow(["대통령", "정책", "정책요약", "시기", "날짜", "기사제목", "url", "감정"])
            
            writer.writerows(new_rows)
        print(f"[Success] {len(new_rows)}개의 새로운 행이 '{filename}'에 성공적으로 추가 저장되었습니다.")
    except Exception as e:
        print(f"[Error] CSV 파일 저장 중 예외 발생: {e}")


def sort_csv_by_date(filename="data/News_Scraping_retouch.csv"):
    """
    저장된 CSV 파일을 날짜 순으로 정렬하여 덮어씁니다.
    날짜 필드(index 4)를 기준으로 오름차순 정렬을 수행합니다.

    Args:
        filename (str): 정렬할 CSV 파일 이름
    """
    if not os.path.exists(filename):
        print(f"[Warning] 정렬할 파일이 존재하지 않습니다: {filename}")
        return

    try:
        # 1. 기존 데이터 읽기
        with open(filename, mode='r', encoding='utf-8-sig') as f:
            reader = csv.reader(f)
            header = next(reader, None)
            if not header:
                print(f"[Info] 정렬 대상 CSV 파일이 비어 있습니다: {filename}")
                return
            rows = list(reader)

        # 2. 날짜 필드(index 4) 기준 정렬
        def parse_date(row):
            if len(row) > 4:
                try:
                    return datetime.strptime(row[4], "%Y-%m-%d").date()
                except ValueError:
                    pass
            return datetime.min.date()

        rows.sort(key=parse_date)

        # 3. 정렬된 데이터로 파일 덮어쓰기
        with open(filename, mode='w', encoding='utf-8-sig', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(header)
            writer.writerows(rows)
            
        print(f"[Success] '{filename}' 파일이 날짜 순으로 정렬되었습니다. (총 {len(rows)}개 행)")
    except Exception as e:
        print(f"[Error] CSV 파일 정렬 중 예외 발생: {e}")


In [50]:
def run_scraping_flow(date_str, filename="News_Scraping.csv"):
    """
    특정 날짜에 대한 전체 스크래핑 및 매핑, CSV 저장 과정을 처리하는 메인 흐름 제어 함수입니다.

    Args:
        date_str (str): YYYYMMDD 또는 YYYY-MM-DD 형식의 날짜 문자열
        filename (str): 저장할 CSV 파일 이름
    """
    # YYYY-MM-DD 또는 YYYYMMDD 포맷 정규화
    clean_date_str = date_str.replace("-", "")
    
    # 1. 정책 정보 매핑 판별
    policy_mappings = get_policy_mapping(clean_date_str)
    
    # 2. 뉴스 스크래핑
    articles = scrape_naver_land_news(clean_date_str)
    if not articles:
        print(f"[Info] {clean_date_str} 날짜에 스크래핑된 뉴스가 없습니다.")
        return
        
    # 3. 데이터 가공
    rows = []
    if policy_mappings:
        for mapping in policy_mappings:
            for title, url in articles:
                rows.append([
                    mapping["president"],
                    mapping["policy"],
                    mapping["policy_summary"],
                    mapping["period"],
                    mapping["date"],
                    title,
                    url,
                    ""
                ])
    else:
        formatted_date = f"{clean_date_str[:4]}-{clean_date_str[4:6]}-{clean_date_str[6:8]}"
        for title, url in articles:
            rows.append([
                "", "", "", "", formatted_date, title, url, ""
            ])
            
    # 4. CSV 저장 및 정렬
    save_to_csv(rows, filename)
    sort_csv_by_date(filename)

In [51]:
def scrape_all_policies(filename="data/News_Scraping_retouch.csv"):
    """
    이재명 대통령 6.27 부동산 대책의 4개 시기별 전체 기간(총 121일)의 모든 날짜 각각에 대해
    하루에 최대 100개씩 뉴스를 스크랩하여 CSV에 저장합니다.
    """
    import time
    total_started = time.time()
    print(f"\\n[Start] 이재명 대통령 6.27 부동산 대책 전체 기간(총 121일) 뉴스 수집을 시작합니다. (하루 100개 기준)")
    
    p = POLICIES[0]
    eff_date = datetime.strptime(p["effective_date"], "%Y-%m-%d").date()
    
    # 각 시기별 날짜 리스트 정의 (시간 역순 탐색)
    periods = {
        "시행전": [eff_date - timedelta(days=i) for i in range(30, 0, -1)],
        "시행일": [eff_date],
        "초기반응": [eff_date + timedelta(days=i) for i in range(30, 0, -1)],  # 30일후 -> 1일후 순
        "체감반응": [eff_date + timedelta(days=i) for i in range(90, 30, -1)]   # 90일후 -> 31일후 순
    }
    
    # 기존 파일 삭제 후 새로 깨끗이 수집
    if os.path.exists(filename):
        try:
            os.remove(filename)
            print(f"[Info] 기존 데이터 파일 '{filename}'을 삭제하고 새로 수집을 시작합니다.")
        except Exception as e:
            print(f"[Warning] 기존 파일 삭제 실패: {e}")
            
    # 수집 진행
    for period_name, date_list in periods.items():
        print(f"\\n>>> [{period_name}] 수집 시작 (기간: {len(date_list)}일, 매일 100개 한도) ...")
        
        for idx, target_date in enumerate(date_list, 1):
            date_str = target_date.strftime("%Y%m%d")
            print(f"[{idx}/{len(date_list)}] {target_date.strftime('%Y-%m-%d')} 수집 중...", end=" ", flush=True)
            
            day_articles = scrape_naver_land_news(date_str, max_articles=100)
            if day_articles:
                print(f"성공: {len(day_articles)}개 수집 완료")
                rows = []
                for title, url in day_articles:
                    rows.append([
                        p["president"],
                        p["policy"],
                        p["summary"],
                        period_name,
                        target_date.strftime("%Y-%m-%d"),
                        title,
                        url,
                        ""
                    ])
                save_to_csv(rows, filename)
            else:
                print("기사 없음")
            
            time.sleep(0.1) # 빠른 수집을 위해 대기 시간 최소화 (0.1초)
            
        print(f"[{period_name}] 전체 수집 완료!")
        
    duration = time.time() - total_started
    print(f"\\n[Finished] 전체 121일 수집 완료. 소요 시간: {duration:.2f}초.")
    sort_csv_by_date(filename)

In [52]:
# 1. 이재명 6.27 대책 전체 기간(총 121일) 동안 매일 100개씩 스크랩 실행
scrape_all_policies(filename="data/News_Scraping_retouch.csv")

\n[Start] 이재명 대통령 6.27 부동산 대책 전체 기간(총 121일) 뉴스 수집을 시작합니다. (하루 100개 기준)
[Info] 기존 데이터 파일 'data/News_Scraping_retouch.csv'을 삭제하고 새로 수집을 시작합니다.
\n>>> [시행전] 수집 시작 (기간: 30일, 매일 100개 한도) ...
[1/30] 2025-05-29 수집 중... 성공: 100개 수집 완료
[Success] 100개의 새로운 행이 'data/News_Scraping_retouch.csv'에 성공적으로 추가 저장되었습니다.
[2/30] 2025-05-30 수집 중... 성공: 100개 수집 완료
[Success] 100개의 새로운 행이 'data/News_Scraping_retouch.csv'에 성공적으로 추가 저장되었습니다.
[3/30] 2025-05-31 수집 중... 성공: 73개 수집 완료
[Success] 73개의 새로운 행이 'data/News_Scraping_retouch.csv'에 성공적으로 추가 저장되었습니다.
[4/30] 2025-06-01 수집 중... 성공: 100개 수집 완료
[Success] 100개의 새로운 행이 'data/News_Scraping_retouch.csv'에 성공적으로 추가 저장되었습니다.
[5/30] 2025-06-02 수집 중... 성공: 100개 수집 완료
[Success] 100개의 새로운 행이 'data/News_Scraping_retouch.csv'에 성공적으로 추가 저장되었습니다.
[6/30] 2025-06-03 수집 중... 성공: 100개 수집 완료
[Success] 100개의 새로운 행이 'data/News_Scraping_retouch.csv'에 성공적으로 추가 저장되었습니다.
[7/30] 2025-06-04 수집 중... 성공: 100개 수집 완료
[Success] 100개의 새로운 행이 'data/News_Scraping_retouch.csv'에 성공적으로 추가 저장되었습니다.
[8/30